# 01. Data collection

**The question.** Can every input the project needs be obtained from public sources, and is each raw file stored unmodified with its provenance?

**Why a portfolio manager cares.** A hedge study is only as good as its inputs. A manager needs to know exactly which prices, rates, and mortgage statistics were used, when they were retrieved, and whether anything was altered.

**Learning objectives**

1. Download (or load) every source with caching and bounded retries.
2. Store each raw file unmodified, with a SHA-256 hash, in a manifest.
3. Record the access status of every source: verified, cached, user-provided, synthetic, or unavailable.
4. Stop clearly if an essential input is missing, and explain the manual route.

| | |
|---|---|
| Inputs | Internet access (or files in `data/local_inputs/`). Smoke mode generates a labeled synthetic world instead. |
| Outputs | `data/raw/**`, `data/raw/manifest.csv`, `results/data_status/access_status.csv` |
| Run after | 00 (optional) |
| Approximate runtime | about 1 minute with downloads, seconds when cached |

## Sources and what was verified before this notebook was written

On 2026-11-02 each source was checked live through a web browser:

| Dataset | Route used here | Verified on 2026-11-02 | Role |
|---|---|---|---|
| Treasury yields DGS10, DGS5, DGS2, DGS3MO | FRED graph CSV, no key | Header `observation_date,<ID>`. Bond-market holidays are present with an empty value (for example 2026-10-12). | Core |
| 30-year mortgage rate MORTGAGE30US | FRED graph CSV | Weekly, dated by the Thursday release. Freddie Mac releases at 12:00 ET Thursday (Wednesday if Thursday is a holiday). Method changed on 2022-11-17. | Core |
| NMDB outstanding-mortgage statistics | FHFA zip of one CSV | 14 columns: SOURCE, FREQUENCY, GEOLEVEL, GEOID, GEONAME, MARKET, PERIOD, YEAR, QUARTER, MONTH, SUPPRESSED, SERIESID, VALUE1, VALUE2. VALUE1 is weighted by loan count, VALUE2 by unpaid balance. Markets: All Mortgages, Enterprise Acquisitions, Government / Non-Conventional, Other Conventional Market. 2013Q1 to 2026Q2, released 2026-09-30. | Core |
| MBB daily closes and distributions | yfinance, then the Yahoo chart API | Chart API returns split-adjusted closes, adjusted closes, and dividend events dated by ex-date. First trade 2007-03-16. | Core |
| VMBS | same | First trade 2009-11-23 | External validation |
| PMBS | same | First trade 2024-09-23 | Descriptive extension |
| PIMCO GNMA and Government Securities Fund, Institutional class (PDMIX) | same | Mutual fund priced at daily NAV from 1997-07-31. All 373 distributions through October 2026 fall on trading days, Yahoo reports no separate capital-gain events, and Yahoo's adjusted close stays within 4 bp a year of the rebuilt total return | PIMCO held-out test |
| PIMCO Mortgage-Backed Securities Fund before its 2024 ETF conversion (PTRIX) | Yahoo | **Unavailable**: Yahoo now maps PTRIX and PMRPX to the PMBS ETF and returns no observations | Optional, user-supplied NAV file only (`tools/convert_pimco_nav.py`) |

These checks verified access and formats. They did not download the full history. That happens when this notebook runs.

In [ ]:
# Setup: find the project root, load the configuration, and pick the run mode.
# The mode comes from MBSHEDGE_MODE (set by run_project.py) or config/project.yaml.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config" / "project.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mbshedge.config import setup, experiment
from mbshedge import plotting as pt

cfg, paths, MODE = setup()
EXP = experiment(cfg, MODE)
pt.setup_style()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [ ]:
from mbshedge import sources as src
from mbshedge import synthetic as syn

store = src.RawStore(paths.raw)
local_dir = paths.local_inputs
refresh = bool(cfg["data"]["refresh"])
start, end = cfg["data"]["start_date"], cfg["data"]["end_date"]
print("raw folder:", paths.rel(paths.raw), "| refresh:", refresh)

## Download or generate

In **smoke mode** this cell builds one synthetic world (seed from the config) and writes it in exactly the raw formats verified above, so every later notebook parses it like real data. Every manifest row is marked `synthetic=True`.

In **development** and **full** mode it downloads each source. A cached file is reused unless `data.refresh` is true, so reruns do not hit the providers again.

In [ ]:
errors = {}
if MODE == "smoke":
    if store.latest("fred_DGS10") is None or store.latest("prices_PDMIX") is None:
        world = syn.make_world(seed=cfg["experiment"]["seed"], scenario="state", end=cfg["smoke"]["synthetic_end_date"])
        syn.write_raw_files(world, store, cfg)
        world["truth"].to_parquet(paths.raw / "synthetic_truth.parquet")
        print("Synthetic world written (seed", cfg["experiment"]["seed"], ")")
    else:
        for d in store.manifest()["dataset"].unique():
            store.add_status(d, "synthetic", "synthetic", "reused synthetic file")
        print("Synthetic world already present, reused.")
else:
    for sid, meta in cfg["data"]["fred_series"].items():
        try:
            df = src.fetch_fred(sid, store, cfg, local_dir, meta.get("start", start), end, refresh, role=meta["role"])
            print(f"{sid:13s} {len(df):6d} rows  {df['date'].min().date()} to {df['date'].max().date()}")
        except src.SourceError as exc:
            errors[sid] = str(exc)
    try:
        nm = src.fetch_nmdb(store, cfg, local_dir, refresh)
        print(f"NMDB          {len(nm):6d} national rows  {nm['period'].min()} to {nm['period'].max()}")
    except src.SourceError as exc:
        errors["NMDB"] = str(exc)
    for tk, meta in cfg["data"]["price_tickers"].items():
        try:
            df = src.fetch_prices(tk, store, cfg, local_dir, start, end, refresh, role=meta["role"])
            print(f"{tk:13s} {len(df):6d} rows  {df['date'].min().date()} to {df['date'].max().date()}")
        except src.SourceError as exc:
            errors[tk] = str(exc)
    src.fetch_document("nmdb_dictionary", cfg["data"]["nmdb"]["dictionary_url"], store, cfg, refresh)
for k, v in errors.items():
    print(f"\nPROBLEM with {k}:\n  {v}")

## Access status of every source

In [ ]:
status = store.status_table()
out_dir = paths.results / "data_status"; out_dir.mkdir(parents=True, exist_ok=True)
status.to_csv(out_dir / "access_status.csv", index=False)
status

## The manifest: provenance of every raw file

Each row is one immutable file: where it came from, when, its SHA-256 hash, size, rows, date span, units, field notes, and the availability rule the pipeline applies later. Recomputing the hashes confirms that no raw file changed after it was saved.

In [ ]:
manifest = store.manifest()
show = ["dataset", "provider", "retrieved_at_utc", "rows", "first_date", "last_date", "sha256", "synthetic"]
display(manifest[show].assign(sha256=manifest["sha256"].str[:12] + "…"))
hashes = store.verify_hashes()
print(f"Hash check: {int(hashes['hash_ok'].sum())} of {len(hashes)} raw files match their manifest hash.")
manifest.to_csv(out_dir / "raw_manifest_snapshot.csv", index=False)

## Essential inputs must be present

The core cannot run without MBB prices, the 10-year and 3-month yields, the PMMS rate, and NMDB. If any is missing the notebook stops here with the manual route. VMBS, PMBS, and PIMCO's GNMA fund (PDMIX) are optional. If one fails, notebook 09 reports it as unavailable and the rest still runs.

In [ ]:
essential = ["prices_MBB", "fred_DGS10", "fred_DGS3MO", "fred_MORTGAGE30US", "fhfa_nmdb_outstanding_national_census"]
have = set(manifest["dataset"])
missing = [d for d in essential if d not in have]
if missing:
    raise RuntimeError(f"Essential inputs missing: {missing}. See the messages above and docs/data_sources.md "
                       "for how to place files in data/local_inputs/.")
optional_missing = [d for d in ["prices_VMBS", "prices_PMBS", "prices_PDMIX", "fred_DGS5", "fred_DGS2"] if d not in have]
print("All essential inputs present." + (f" Optional inputs missing: {optional_missing}" if optional_missing else ""))